In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, time, numpy as np, pandas as pd
import torch
from transformers import AutoModel
import soundfile as sf  # Reliable audio loading
import subprocess

# Install librosa if needed
try:
    import librosa
except ImportError:
    subprocess.run(['pip', 'install', 'librosa', '-q'])
    import librosa

BASE = '/content/drive/MyDrive/standup4ai'
AUDIO_DIR = BASE + '/audio_1000'
LABEL_DIR = BASE + '/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train'
OUT_DIR = BASE + '/wordlevel_wavlm_features'
CKPT_FILE = BASE + '/wordlevel_checkpoint.json'
os.makedirs(OUT_DIR, exist_ok=True)

# Load checkpoint
done = set()
if os.path.exists(CKPT_FILE):
    with open(CKPT_FILE) as f:
        done = set(json.load(f).get('done', []))

# Build audio lookup with fallback for ,lang suffix files
audio_exts = ('.m4a', '.wav', '.mp3', '.webm')
available_audio = {}
if os.path.exists(AUDIO_DIR):
    for f in os.listdir(AUDIO_DIR):
        if not any(f.endswith(e) for e in audio_exts):
            continue
        vid_base = f.rsplit('.', 1)[0]  # e.g. '18H1aeoGybw' or '1Vp9LRG865c,pt'
        # Try direct match first
        if vid_base not in available_audio:
            available_audio[vid_base] = os.path.join(AUDIO_DIR, f)
        # Try without ,lang suffix
        if ',' in vid_base:
            vid_clean = vid_base.split(',')[0]
            if vid_clean not in available_audio:
                available_audio[vid_clean] = os.path.join(AUDIO_DIR, f)

available_labels = set()
if os.path.exists(LABEL_DIR):
    for f in os.listdir(LABEL_DIR):
        if f.endswith('.csv'):
            available_labels.add(f.replace('.csv', ''))

overlap = sorted(available_audio.keys() & available_labels - done)
print(f'Audio: {len(available_audio)} | Labels: {len(available_labels)} | To process: {len(overlap)}')
print(f'Already done: {len(done)}')
if overlap:
    print(f'Sample: {overlap[:3]}')


In [ ]:
# Load WavLM on GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Loading WavLM-base...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
print('WavLM loaded!')


In [ ]:
# Per-word feature extraction (fixed audio loading)
SR = 16000
SAMPLE_DURATION = 0.02  # 20ms minimum for WavLM

def parse_timestamp(ts_str):
    ts_str = str(ts_str).strip()
    try:
        p = ts_str.strip('[]').split(',')
        return float(p[0]), float(p[1])
    except:
        return None, None

def load_audio_segment(audio_path, t0, t1, target_sr=16000):
    """Load audio segment robustly, converting to mono.
    
    Tries soundfile first (fast), falls back to librosa.
    Returns mono audio at target_sr sample rate.
    """
    try:
        # soundfile is fast and handles most formats
        y, sr = sf.read(audio_path, start=int(t0 * target_sr),
                       stop=int(t1 * target_sr), dtype='float32')
        if len(y.shape) > 1:  # multi-channel -> mono
            y = y.mean(axis=1)
        if sr != target_sr:
            # Resample
            import librosa
            y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
        return y
    except Exception as e:
        # Fallback to librosa
        import librosa
        try:
            y, sr = librosa.load(audio_path, offset=t0, duration=t1-t0,
                             sr=target_sr, mono=True)
            return y
        except:
            # Last resort: load full file and slice
            y_full, sr = librosa.load(audio_path, sr=target_sr, mono=True)
            s, e = int(t0 * target_sr), min(int(t1 * target_sr), len(y_full))
            return y_full[s:e]

def extract_word_features(audio_path, word_times, batch_size=32):
    """Extract WavLM embedding for each word using exact timestamps.
    
    For each word [t0, t1]:
    1. Load audio segment at 16kHz mono (using soundfile)
    2. Pad to minimum 0.02s if too short
    3. Run WavLM on the segment (batched)
    4. Mean-pool over time -> 768-dim embedding
    """
    n_words = len(word_times)
    if n_words == 0:
        return None
    
    features = []
    batch_size = 32  # Smaller batch for memory safety
    
    for batch_start in range(0, n_words, batch_size):
        batch_times = word_times[batch_start:batch_start + batch_size]
        segments, valid_mask = [], []
        
        for t0, t1 in batch_times:
            dur = t1 - t0
            if dur <= 0:
                segments.append(np.zeros(int(SAMPLE_DURATION * SR), dtype=np.float32))
                valid_mask.append(False)
                continue
            
            try:
                seg = load_audio_segment(audio_path, t0, t1, SR)
                min_len = int(SAMPLE_DURATION * SR)
                if len(seg) < min_len:
                    seg = np.pad(seg, (0, min_len - len(seg)))
                segments.append(seg.astype(np.float32))
                valid_mask.append(True)
            except Exception as e:
                segments.append(np.zeros(int(SAMPLE_DURATION * SR), dtype=np.float32))
                valid_mask.append(False)
        
        if not segments:
            continue
        
        # Stack into batch
        try:
            batch = torch.tensor(np.stack(segments), dtype=torch.float32).to(device)
        except Exception as e:
            # Handle variable length segments by padding
            max_len = max(len(s) for s in segments)
            padded = []
            for s in segments:
                if len(s) < max_len:
                    s = np.pad(s, (0, max_len - len(s)))
                padded.append(s)
            batch = torch.tensor(np.stack(padded), dtype=torch.float32).to(device)
        
        with torch.no_grad():
            out = wavlm(batch).last_hidden_state  # (batch, seq, 768)
            emb = out.mean(dim=1).squeeze(1)       # (batch, 768)
        
        for i, v in enumerate(valid_mask):
            if v:
                features.append(emb[i].cpu().numpy())
            else:
                features.append(np.zeros(768, dtype=np.float32))
    
    return np.array(features, dtype=np.float32)  # (n_words, 768)

print('Feature extractor ready')


In [ ]:
# Process all videos
SKIP_THRESHOLD = 50

t0 = time.time()
for i, vid in enumerate(overlap):
    out_file = os.path.join(OUT_DIR, vid + '_word_features.npy')
    if os.path.exists(out_file):
        continue
    
    audio_path = available_audio[vid]
    label_path = os.path.join(LABEL_DIR, vid + '.csv')
    
    # Load labels
    df = pd.read_csv(label_path)
    if len(df) < SKIP_THRESHOLD:
        done.add(vid)
        continue
    
    # Get word timestamps AND labels together (aligned!)
    word_times, word_labels = [], []
    for _, row in df.iterrows():
        t0_w, t1_w = parse_timestamp(row['timestamp'])
        if t0_w is not None:
            word_times.append((t0_w, t1_w))
            word_labels.append(str(row['label']).strip())
    
    if len(word_times) < SKIP_THRESHOLD:
        done.add(vid)
        continue
    
    # Extract features
    try:
        feats = extract_word_features(audio_path, word_times)
    except Exception as e:
        print(f'ERROR {vid}: {e}')
        continue
    
    if feats is None or len(feats) == 0:
        continue
    
    # Verify alignment
    assert len(feats) == len(word_labels), f'{vid}: {len(feats)} feats != {len(word_labels)} labels'
    
    # Save
    np.save(out_file, feats)
    done.add(vid)
    
    elapsed = time.time() - t0
    rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
    
    print(f'{i+1}/{len(overlap)} {vid}: {feats.shape} | done={len(done)} | rate={rate:.0f}/hr')
    
    if len(done) % 10 == 0:
        with open(CKPT_FILE, 'w') as f:
            json.dump({'done': list(done)}, f)

with open(CKPT_FILE, 'w') as f:
    json.dump({'done': list(done)}, f)

print(f'\nDone: {len(done)}/{len(overlap)} videos in {(time.time()-t0)/60:.0f} min')


In [ ]:
# Summary
OUT_DIR = '/content/drive/MyDrive/standup4ai/wordlevel_wavlm_features'
feat_files = sorted([f for f in os.listdir(OUT_DIR) if f.endswith('_word_features.npy')])
print(f'Word-level features: {len(feat_files)} videos')
for f in feat_files[:5]:
    d = np.load(os.path.join(OUT_DIR, f))
    print(f'  {f}: {d.shape}')
